In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
from tools import ActivationExtractor, plot_activations, plot_activation_stats

/workspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32 if device == "cpu" else torch.float16,
)
model = model.to(device)

Using device: cpu


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 764.12it/s, Materializing param=h.0.attn.bias]                     
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Define which layers/components to extract
layers_to_extract = ["transformer.h.0", "transformer.h.11"]

# Initialize the extractor tool. Hooks are persisted until `extractor.clear_hooks()`
# is called, so there's no option to toggle persistence per-call.
extractor = ActivationExtractor(model, tokenizer, layers_to_extract)

# Example: Extract activations
prompt = "The future of AI is"
print(f"Analyzing prompt: '{prompt}'")

results = extractor.extract(prompt)

tokens = results["tokens"]
activations = results["activations"]

print(f"Tokens: {tokens}")

for layer, act in activations.items():
    print(f"\nLayer: {layer}")
    print(f"Activation shape: {act.shape}")
    # Print first few values for the last token
    print(f"Values for last token '{tokens[-1]}': {act[0, -1, :5]}...")

# Optional: persist model state and hooks metadata to disk. Hooks metadata
# (e.g. layer names) is saved to a small JSON file; actual hook objects are
# not pickled because they are typically not serializable.
# extractor.save_model_with_hooks("./saved_model", save_tokenizer=True)


Analyzing prompt: 'The future of AI is'
Tokens: ['The', 'Ġfuture', 'Ġof', 'ĠAI', 'Ġis']

Layer: transformer.h.0
Activation shape: torch.Size([1, 5, 768])
Values for last token 'Ġis': tensor([-1.7169,  0.3107, -0.2254, -0.3152,  0.7702])...

Layer: transformer.h.11
Activation shape: torch.Size([1, 5, 768])
Values for last token 'Ġis': tensor([ 4.4027,  9.4341,  1.6082, -2.6396,  0.9568])...


## Text Generation

Now let's use the `generate()` method to create new text:

In [4]:
# Generate text using the same extractor
prompt = "The future of AI is"
print(f"Generating from prompt: '{prompt}'\n")

# Generate with sampling (creative output)
result = extractor.generate(
    text=prompt,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

print(f"Generated text:\n{result['generated_text']}")
print(f"\n{'='*60}")
print(f"Prompt tokens: {len(result['full_tokens']) - len(result['generated_tokens'])}")
print(f"Generated tokens: {len(result['generated_tokens'])}")
print(f"\nGenerated tokens: {result['generated_tokens'][:10]}...")  # Show first 10

Generating from prompt: 'The future of AI is'

Generated text:
The future of AI is also cloud-based. AI is being built into the world of social media and is already being used to predict the future.

The main aim of the "Federation of Artificial Intelligence (AI)" is to create a unified global culture of social

Prompt tokens: 5
Generated tokens: 50

Generated tokens: ['Ġalso', 'Ġcloud', '-', 'based', '.', 'ĠAI', 'Ġis', 'Ġbeing', 'Ġbuilt', 'Ġinto']...


### Comparison: Different Generation Strategies

In [ ]:
# Compare different generation strategies
prompt = "Once upon a time"

strategies = [
    {"name": "Greedy (deterministic)", "params": {"do_sample": False, "max_new_tokens": 30}},
    {"name": "High temperature (creative)", "params": {"temperature": 1.5, "max_new_tokens": 30}},
    {"name": "Low temperature (focused)", "params": {"temperature": 0.5, "max_new_tokens": 30}},
    {"name": "Top-k sampling", "params": {"top_k": 10, "temperature": 0.8, "max_new_tokens": 30}},
]

print(f"Prompt: '{prompt}'\n")
for strategy in strategies:
    result = extractor.generate(text=prompt, **strategy["params"])
    print(f"{strategy['name']}:")
    print(f"  → {result['generated_text']}")
    print()

### Generation with Activation Collection

Collect activations at each generation step to see how the model's internal state evolves:

In [ ]:
# Generate text while collecting activations at each step
# Note: activations are always collected during generation
prompt = "The cat sat on the"
result = extractor.generate(
    text=prompt,
    max_new_tokens=8,
    temperature=0.7,
    do_sample=True
)

print(f"Prompt: '{prompt}'")
print(f"Generated: '{result['generated_text']}'")
print(f"\nGenerated tokens: {result['generated_tokens']}")
print(f"\nActivations collected for {len(result['activations'])} generation steps")

# Analyze activations at each generation step
for step, (token, step_activations) in enumerate(zip(result['generated_tokens'], result['activations'])):
    print(f"\nStep {step + 1} - Token: '{token}'")
    for layer_name, act in step_activations.items():
        # Get stats for the last token's activation in this step
        last_token_act = act[0, -1, :]
        print(f"  {layer_name}: shape={act.shape}, mean={last_token_act.mean():.3f}, std={last_token_act.std():.3f}")

### Activation Visualization

Visualize the activation patterns using built-in plotting utilities:

In [ ]:
# Visualize activations from extraction
prompt = "The future of AI is"
results = extractor.extract(prompt)

# Plot activation heatmaps
fig = plot_activations(results["activations"], results["tokens"])
plt.show()

# Get activation statistics
stats = extractor.get_activation_stats(results["activations"])
for layer, layer_stats in stats.items():
    print(f"{layer}: mean={layer_stats['mean']:.3f}, std={layer_stats['std']:.3f}")

In [ ]:
# Visualize activation statistics over generation steps
prompt = "Once upon a time"
result = extractor.generate(
    text=prompt,
    max_new_tokens=10,
    temperature=0.7,
    do_sample=True
)

print(f"Generated: {result['generated_text']}")

# Plot activation statistics over generation steps
fig = plot_activation_stats(result["activations"], result["generated_tokens"])
plt.show()